In [2]:
import numpy as np
import pandas as pd

# ==========================================
# Synthetic Core Capillary Pressure Dataset
# ==========================================
np.random.seed(42)

NUM_CURVES = 5000      
NUM_POINTS = 10        

def porosity_permeability(phi, scatter_std, k_min, k_max):
    log_k = -1.0 + 12.0 * phi + np.random.normal(0, scatter_std)
    k = 10 ** log_k
    return np.clip(k, k_min, k_max)

dataset = []
lithologies = ['Sandstone', 'Carbonate', 'Shaly-Sand']
lith_multipliers = {'Sandstone': 1.0, 'Carbonate': 1.25, 'Shaly-Sand': 0.85}

for curve in range(1, NUM_CURVES + 1):
    # Core & Reservoir Properties
    core_porosity = np.random.uniform(0.08, 0.30)
    core_perm = porosity_permeability(core_porosity, scatter_std=0.55, k_min=0.05, k_max=6000)
    
    reservoir_porosity = np.clip(core_porosity + np.random.normal(0, 0.04), 0.05, 0.38)
    reservoir_perm = porosity_permeability(reservoir_porosity, scatter_std=0.75, k_min=0.1, k_max=10000)

    # Lithology and Heterogeneity Setup
    lithology = np.random.choice(lithologies)
    lith_factor = lith_multipliers[lithology]
    
    j_ratio_core = np.sqrt(core_perm / core_porosity)
    j_ratio_res = np.sqrt(reservoir_perm / reservoir_porosity)
    
    # Target Scaling Factor (Base formula * Lithology factor * Heterogeneity noise)
    heterogeneity_noise = np.random.normal(1.0, 0.05) 
    target_scaling_factor = (j_ratio_core / j_ratio_res) * lith_factor * heterogeneity_noise

    # Brooks-Corey Parameters
    entry_pressure = np.clip((120.0 / j_ratio_core) * np.random.lognormal(mean=0.0, sigma=0.25), 1.0, 300.0)
    lam = np.clip(1.0 + 3.0 * (core_porosity - 0.08) / (0.30 - 0.08) + np.random.normal(0, 0.35), 0.6, 5.0)

    Sw = np.sort(np.random.uniform(0.15, 0.95, NUM_POINTS))
    Se = np.clip((Sw - Sw.min()) / (Sw.max() - Sw.min()), 0.02, 0.99)
    Pc = entry_pressure * (Se ** (-1 / lam))
    Pc = np.maximum(Pc + np.random.normal(0, 0.03 * Pc), 0.5)
    Pc = np.sort(Pc)[::-1]

    # Save Row
    row = {
        "Curve_ID": curve,
        "Lithology": lithology,
        "Target_Scaling_Factor": target_scaling_factor,
        "Core_Porosity": round(core_porosity, 4),
        "Core_Permeability_mD": round(core_perm, 3),
        "Reservoir_Porosity": round(reservoir_porosity, 4),
        "Reservoir_Permeability_mD": round(reservoir_perm, 3)
    }

    for i in range(NUM_POINTS):
        row[f"Sw_{i+1}"] = round(Sw[i], 4)
    for i in range(NUM_POINTS):
        row[f"Pc_{i+1}"] = round(Pc[i], 3)

    dataset.append(row)

# Save CSV
df = pd.DataFrame(dataset)
df.to_csv("synthetic_core_pc_dataset.csv", index=False)
print("Dataset Generated Successfully!\nTotal Curves :", len(df))

Dataset Generated Successfully!
Total Curves : 5000
